# Datasets for the multimodal integration chapters

The multimodal integration chapters use the bone marrow mononuclear cells of the NeurIPS 2021 multimodal integration challenge, which were measured with CITE-seq (RNA and surface proteins) and with 10x Multiome (RNA and chromatin accessibility) at four sites.
We store each dataset as a MuData object with raw counts and add a coarse annotation shared by both datasets, since the two were annotated at different resolutions.
The GENCODE gene annotation provides the gene coordinates that GLUE needs to link genes and peaks.

In [1]:
import lamindb as ln

ln.connect("theislab/sc-best-practices")

ln.track("NgNjOMSxsQMG")

→ connected lamindb: theislab/sc-best-practices


! It looks like you are running jupyter lab but don't have jupyter-server module installed. Please install it via pip install jupyter-server


→ found notebook multimodal_integration_dataset.ipynb, making new version


→ created Transform('NgNjOMSxsQMG0001', key='multimodal_integration_dataset.ipynb'), started new Run('uoyhLsH2BtmQl8Oq') at 2026-09-29 21:36:22 UTC


→ notebook imports: anndata==0.13.2 lamindb-core==2.10.0 mudata==0.4.1 pooch==1.9.0


In [2]:
import anndata as ad
import mudata as md
import pooch

In [3]:
geo = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE194nnn/GSE194122/suppl"
cite_path = pooch.retrieve(
    f"{geo}/GSE194122_openproblems_neurips2021_cite_BMMC_processed.h5ad.gz",
    known_hash="sha256:b9b50fade9349719cba23c97c6515d3501a32ee3735fe95fe51221d2e8a5f361",
    processor=pooch.Decompress(),
)
multiome_path = pooch.retrieve(
    f"{geo}/GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad.gz",
    known_hash="sha256:53b516e7a35518e7678f90a2f27867ae2a0054dba6ac0c52fafb67b5a80f415a",
    processor=pooch.Decompress(),
)
gtf_path = pooch.retrieve(
    "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_41/gencode.v41.basic.annotation.gtf.gz",
    known_hash="sha256:5d226e52471e8a5de7090739b45c25c7045378c1f1d207d224dbd59dfb710704",
)

The coarse annotation groups the cell types of both datasets into lineages.
Proteins are prefixed with `PROT_`, since many share their name with the gene that encodes them.

In [4]:
LINEAGE = {
    "B1 B": "B",
    "B1 B IGKC+": "B",
    "B1 B IGKC-": "B",
    "Naive CD20+ B": "B",
    "Naive CD20+ B IGKC+": "B",
    "Naive CD20+ B IGKC-": "B",
    "Transitional B": "B",
    "Plasma cell": "Plasma cell",
    "Plasma cell IGKC+": "Plasma cell",
    "Plasma cell IGKC-": "Plasma cell",
    "Plasmablast IGKC+": "Plasma cell",
    "Plasmablast IGKC-": "Plasma cell",
    "CD4+ T activated": "CD4+ T",
    "CD4+ T naive": "CD4+ T",
    "CD4+ T CD314+ CD45RA+": "CD4+ T",
    "CD4+ T activated integrinB7+": "CD4+ T",
    "T reg": "CD4+ T",
    "CD8+ T": "CD8+ T",
    "CD8+ T naive": "CD8+ T",
    "CD8+ T CD49f+": "CD8+ T",
    "CD8+ T CD57+ CD45RA+": "CD8+ T",
    "CD8+ T CD57+ CD45RO+": "CD8+ T",
    "CD8+ T CD69+ CD45RA+": "CD8+ T",
    "CD8+ T CD69+ CD45RO+": "CD8+ T",
    "CD8+ T TIGIT+ CD45RA+": "CD8+ T",
    "CD8+ T TIGIT+ CD45RO+": "CD8+ T",
    "CD8+ T naive CD127+ CD26- CD101-": "CD8+ T",
    "MAIT": "Other T",
    "dnT": "Other T",
    "gdT CD158b+": "Other T",
    "gdT TCRVD2+": "Other T",
    "T prog cycling": "Other T",
    "NK": "NK and ILC",
    "NK CD158e1+": "NK and ILC",
    "ILC": "NK and ILC",
    "ILC1": "NK and ILC",
    "CD14+ Mono": "Monocyte",
    "CD16+ Mono": "Monocyte",
    "cDC1": "Dendritic cell",
    "cDC2": "Dendritic cell",
    "pDC": "Dendritic cell",
    "Erythroblast": "Erythroid",
    "Normoblast": "Erythroid",
    "Proerythroblast": "Erythroid",
    "Reticulocyte": "Erythroid",
    "MK/E prog": "Erythroid",
    "HSC": "Progenitor",
    "G/M prog": "Progenitor",
    "Lymph prog": "Progenitor",
    "ID2-hi myeloid prog": "Progenitor",
}

In [5]:
def to_mudata(path: str, modalities: dict[str, str]) -> md.MuData:
    adata = ad.read_h5ad(path)
    adata.X = adata.layers.pop("counts")
    adata.obs = adata.obs[["cell_type", "batch", "Site", "DonorNumber"]].rename(
        columns={"Site": "site", "DonorNumber": "donor"}
    )
    adata.obs["cell_type_l1"] = adata.obs["cell_type"].map(LINEAGE).astype("category")
    adata.obsm = {}
    adata.uns = {}
    mods = {}
    for name, feature_type in modalities.items():
        mod = adata[:, adata.var["feature_types"] == feature_type].copy()
        mod.var = mod.var[["gene_id"]] if name == "rna" else mod.var[[]]
        if name == "adt":
            mod.var_names = "PROT_" + mod.var_names
        mods[name] = mod
    return md.MuData(mods)

In [6]:
cite = to_mudata(cite_path, {"rna": "GEX", "adt": "ADT"})
multiome = to_mudata(multiome_path, {"rna": "GEX", "atac": "ATAC"})
cite, multiome

/tmp/ipykernel_2566645/756022207.py:2: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  adata = ad.read_h5ad(path)


(MuData object with n_obs × n_vars = 90261 × 14087
   2 modalities
     rna:	90261 × 13953
       obs:	'cell_type', 'batch', 'site', 'donor', 'cell_type_l1'
       var:	'gene_id'
       layers:	None
     adt:	90261 × 134
       obs:	'cell_type', 'batch', 'site', 'donor', 'cell_type_l1'
       layers:	None,
 MuData object with n_obs × n_vars = 69249 × 129921
   2 modalities
     rna:	69249 × 13431
       obs:	'cell_type', 'batch', 'site', 'donor', 'cell_type_l1'
       var:	'gene_id'
       layers:	None
     atac:	69249 × 116490
       obs:	'cell_type', 'batch', 'site', 'donor', 'cell_type_l1'
       layers:	None)

In [7]:
ln.Artifact.from_mudata(
    cite,
    key="multimodal_integration/neurips2021_cite.h5mu",
    description="NeurIPS 2021 bone marrow CITE-seq with raw RNA and protein counts",
).save()
ln.Artifact.from_mudata(
    multiome,
    key="multimodal_integration/neurips2021_multiome.h5mu",
    description="NeurIPS 2021 bone marrow Multiome with raw RNA and ATAC counts",
).save()
ln.Artifact(
    gtf_path,
    key="multimodal_integration/gencode.v41.basic.annotation.gtf.gz",
    description="GENCODE v41 basic gene annotation of GRCh38",
).save()

→ creating new artifact version for key 'multimodal_integration/neurips2021_cite.h5mu' in storage 's3://lamin-eu-central-1/VPwcjx3CDAa2'


... uploading agZ4ai8NKFeKEfwo0001.h5mu:  0.0%

... uploading agZ4ai8NKFeKEfwo0001.h5mu:  4.6%

... uploading agZ4ai8NKFeKEfwo0001.h5mu:  9.1%

... uploading agZ4ai8NKFeKEfwo0001.h5mu: 18.2%

...

... uploading QqPE4qYNdp2u6biw0001.h5mu: 95.5%

... uploading QqPE4qYNdp2u6biw0001.h5mu: 99.5%

... uploading QqPE4qYNdp2u6biw0001.h5mu: 100.0%
• replacing the existing cache path /home/lheumos/.cache/lamindb/lamin-eu-central-1/VPwcjx3CDAa2/multimodal_integration/neurips2021_multiome.h5mu


→ returning artifact with same hash: Artifact(uid='8LJYO3Y5rBexkIol0000', key='multimodal_integration/gencode.v41.basic.annotation.gtf.gz', description='GENCODE v41 basic gene annotation of GRCh38', suffix='.gtf.gz', kind=None, otype=None, size=29161889, hash='T7y6J0JaZBG0saH4RlUYeA', n_files=None, n_observations=None, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=189, schema_id=None, created_by_id=2, created_at=2026-09-29 21:35:58 UTC, is_locked=False, version_tag=None, is_latest=True); to track this artifact as an input, use: ln.Artifact.get()


Artifact(uid='8LJYO3Y5rBexkIol0000', key='multimodal_integration/gencode.v41.basic.annotation.gtf.gz', description='GENCODE v41 basic gene annotation of GRCh38', suffix='.gtf.gz', kind=None, otype=None, size=29161889, hash='T7y6J0JaZBG0saH4RlUYeA', n_files=None, n_observations=None, extra_data=None, branch_id=1, created_on_id=1, space_id=1, storage_id=1, run_id=189, schema_id=None, created_by_id=2, created_at=2026-09-29 21:35:58 UTC, is_locked=False, version_tag=None, is_latest=True)

In [8]:
ln.finish()